In [5]:
MTGFLOW_REPO_PATH = "./MTGFLOW-main"

In [7]:
import os, sys

MTGFLOW_REPO_PATH = "./MTGFLOW-main"

print("Notebook CWD:", os.getcwd())
print("MTGFLOW_REPO_PATH:", MTGFLOW_REPO_PATH)
print("Exists?", os.path.exists(MTGFLOW_REPO_PATH))

# Add to python path + test import
if MTGFLOW_REPO_PATH not in sys.path:
    sys.path.insert(0, MTGFLOW_REPO_PATH)

from models.MTGFLOW import MTGFLOW
print("✅ MTGFLOW import works:", MTGFLOW)

Notebook CWD: /home/ec2-user/mtad-gat-independent
MTGFLOW_REPO_PATH: ./MTGFLOW-main
Exists? True
✅ MTGFLOW import works: <class 'models.MTGFLOW.MTGFLOW'>


In [2]:
import sys
!{sys.executable} -m pip install matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 156.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 183.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 111.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [matplotlib]7 [matplotlib]


In [1]:
import os, sys

MTAD_TOOLS_PATH = "/home/ec2-user/mtad-gat-independent"   # adjust if needed
MTGFLOW_REPO_PATH = "/home/ec2-user/mtad-gat-independent/MTGFLOW-main"  # adjust if needed

assert os.path.isdir(MTAD_TOOLS_PATH), MTAD_TOOLS_PATH
assert os.path.isdir(MTGFLOW_REPO_PATH), MTGFLOW_REPO_PATH

# Remove both paths if already present (avoid duplicates)
sys.path = [p for p in sys.path if p not in [MTAD_TOOLS_PATH, MTGFLOW_REPO_PATH]]

# Put mtad-tools FIRST so "utils" resolves to mtad-gat-independent/utils.py
sys.path.insert(0, MTAD_TOOLS_PATH)

# Put MTGFlow AFTER
sys.path.insert(1, MTGFLOW_REPO_PATH)

print("sys.path[0] =", sys.path[0])
print("sys.path[1] =", sys.path[1])

# Now imports will resolve correctly
from dataloader import load_dataset
from data_preprocess import generate_windows

# And MTGFlow import will still work
from models.MTGFLOW import MTGFLOW

print("✅ Imports OK (mtad-tools utils + MTGFlow)")

sys.path[0] = /home/ec2-user/mtad-gat-independent
sys.path[1] = /home/ec2-user/mtad-gat-independent/MTGFLOW-main


2026-03-04 05:31:33.549793: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


✅ Imports OK (mtad-tools utils + MTGFlow)


In [8]:

# %%
# 0) Imports and environment setup

import os
import sys
import math
import random
import logging
import warnings
import yaml
import numpy as np

import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)


Using device: cuda



## 1) Paths and config selection

These mirror the way your MTAD-GAT notebook loads YAML configs.

- `EXP_ID` is typically one of the experiment keys in `mtad_gat.yaml` (e.g., `mtad_gat_SMD`).
- `DATASET_CFG_FILE` is your dataset YAML (e.g., `SMD.yaml`).

If you want to run a different dataset (WADI, SWaT, etc.), use the corresponding YAML you already have.


In [9]:

# %%
# 1) Config selection (edit these)

EXP_ID = "mtad_gat_SMD"
DATASET_CFG_FILE = "SMD.yaml"

# Path to MTGFlow repo (expected to be next to this notebook)
MTGFLOW_REPO_PATH = "./MTGFLOW-main"   # <-- change if needed

# For SMD, mtad-tools often loads a list of entities (machines).
# MTGFlow is multivariate; we typically train per-entity (machine) treating features as sensors.
ENTITY_INDEX = 0  # choose which entity to run, e.g. 0 for machine-1, 1 for machine-2, etc.

# Training hyperparameters for MTGFlow (you can change freely)
MTGFLOW_EPOCHS = 40
MTGFLOW_LR = 2e-3
MTGFLOW_WEIGHT_DECAY = 5e-4
MTGFLOW_BATCH_SIZE = 256
GRAD_CLIP = 1.0

print("EXP_ID:", EXP_ID)
print("DATASET_CFG_FILE:", DATASET_CFG_FILE)
print("MTGFLOW_REPO_PATH:", MTGFLOW_REPO_PATH)


EXP_ID: mtad_gat_SMD
DATASET_CFG_FILE: SMD.yaml
MTGFLOW_REPO_PATH: ./MTGFLOW-main



## 2) Load YAML configs (same pattern as MTAD-GAT)

This cell reads:
- model config from `mtad_gat.yaml` (to reuse `window_size`, `stride`, etc.)
- dataset config from `SMD.yaml` (to reuse `data_root`, `entities`, etc.)


In [10]:

# %%
def load_model_config(model_cfg_path="mtad_gat.yaml", exp_id="mtad_gat_SMD"):
    with open(model_cfg_path, "r") as f:
        cfg_all = yaml.safe_load(f)

    if "Base" in cfg_all:
        base = cfg_all["Base"] or {}
    else:
        base = {}

    if exp_id not in cfg_all:
        raise ValueError(f"Experiment id '{exp_id}' not found. Available: {list(cfg_all.keys())}")

    merged = dict(base)
    merged.update(cfg_all[exp_id] or {})
    return merged

def load_dataset_config(dataset_cfg_path="SMD.yaml"):
    with open(dataset_cfg_path, "r") as f:
        return yaml.safe_load(f)

model_params = load_model_config("mtad_gat.yaml", EXP_ID)
dataset_params = load_dataset_config(DATASET_CFG_FILE)

print("=== Model params (subset) ===")
for k in ["dataset_id", "batch_size", "window_size", "stride", "nb_epoch"]:
    if k in model_params:
        print(f"{k}: {model_params[k]}")

print("\n=== Dataset params (subset) ===")
for k in ["dataset", "data_root", "dim", "entities"]:
    if k in dataset_params:
        v = dataset_params[k]
        if k == "entities" and isinstance(v, list):
            print(f"{k}: list(len={len(v)}) first5={v[:5]}")
        else:
            print(f"{k}: {v}")


=== Model params (subset) ===
dataset_id: SMD
batch_size: 512
window_size: 100
stride: 1
nb_epoch: 10

=== Dataset params (subset) ===


In [11]:
# --- Debug what we loaded ---
print("dataset_params keys:", list(dataset_params.keys())[:50])

# --- Unwrap if nested like {"SMD": {...}} or {"Base": {...}, "SMD": {...}} ---
if "entities" not in dataset_params:
    # If it's a 2-level structure with Base + dataset_name
    if "Base" in dataset_params and DATASET_NAME in dataset_params:
        merged = {}
        merged.update(dataset_params.get("Base", {}))
        merged.update(dataset_params.get(DATASET_NAME, {}))
        dataset_params = merged
        print("Unwrapped Base + dataset section. New keys:", list(dataset_params.keys())[:50])

    # If it's a 1-key wrapper like {"SMD": {...}}
    elif len(dataset_params) == 1 and isinstance(next(iter(dataset_params.values())), dict):
        dataset_params = next(iter(dataset_params.values()))
        print("Unwrapped single nested dataset. New keys:", list(dataset_params.keys())[:50])

# --- If still no entities, auto-create it (single-entity datasets like SWaT) ---
if "entities" not in dataset_params:
    # Use dataset name as a default entity id
    dataset_params["entities"] = [DATASET_NAME]
    print("No 'entities' in config. Set default entities =", dataset_params["entities"])

# Safety: show final
print("Final entities count:", len(dataset_params["entities"]))
print("First entities:", dataset_params["entities"][:5])

dataset_params keys: ['Base', 'SMD']


NameError: name 'DATASET_NAME' is not defined

In [12]:
import os, yaml

print("Current working dir:", os.getcwd())

# If you have these already, print them
print("DATASET_CFG_FILE:", DATASET_CFG_FILE)

# Load dataset YAML
with open(DATASET_CFG_FILE, "r") as f:
    dataset_params_raw = yaml.safe_load(f)

print("Top-level keys in dataset YAML:", list(dataset_params_raw.keys())[:20])

# Determine dataset name (try multiple fallbacks)
DATASET_NAME = dataset_params_raw.get("dataset", None)
if DATASET_NAME is None:
    # if yaml is nested: {Base:..., SMD:...} pick non-Base key
    keys = [k for k in dataset_params_raw.keys() if k.lower() != "base"]
    DATASET_NAME = keys[0] if len(keys) == 1 else "UNKNOWN_DATASET"

print("Resolved DATASET_NAME =", DATASET_NAME)

# Normalize dataset_params into a flat dict
dataset_params = dataset_params_raw
if "Base" in dataset_params_raw and DATASET_NAME in dataset_params_raw:
    dataset_params = {}
    dataset_params.update(dataset_params_raw.get("Base", {}))
    dataset_params.update(dataset_params_raw.get(DATASET_NAME, {}))
elif isinstance(dataset_params_raw.get(DATASET_NAME, None), dict) and "data_root" not in dataset_params_raw:
    # single nested section
    dataset_params = dataset_params_raw[DATASET_NAME]

print("Normalized dataset_params keys:", list(dataset_params.keys())[:30])

# Resolve data_root (required)
def pick_first(d, candidates):
    for k in candidates:
        if k in d and d[k] is not None:
            return d[k]
    return None

data_root = pick_first(dataset_params, ["data_root", "root_dir", "data_dir", "root", "path", "data_path"])
if data_root is None:
    raise KeyError(f"Could not find data_root in dataset_params. Keys={list(dataset_params.keys())}")

print("Resolved data_root =", data_root)

# Ensure entities exists (some datasets define none)
if "entities" not in dataset_params:
    dataset_params["entities"] = [DATASET_NAME]
    print("No entities in YAML -> set default entities =", dataset_params["entities"])
else:
    print("Entities count:", len(dataset_params['entities']), "first:", dataset_params['entities'][:5])

Current working dir: /home/ec2-user/mtad-gat-independent
DATASET_CFG_FILE: SMD.yaml
Top-level keys in dataset YAML: ['Base', 'SMD']
Resolved DATASET_NAME = SMD
Normalized dataset_params keys: ['dataset', 'data_root', 'model_root', 'benchmark_dir', 'train_postfix', 'test_postfix', 'test_label_postfix', 'dim', 'nrows', 'entities', 'valid_ratio']
Resolved data_root = ../data/SMD/
Entities count: 28 first: ['machine-1-1', 'machine-1-2', 'machine-1-3', 'machine-1-4', 'machine-1-5']



## 3) Load data and generate windows (mtad-tools)

Stage A (Step 3): Build windows cache per entity
Input: raw SMD files
Output per entity: cache_windows_smd/{entity}_T{T}_S{S}.npz

train_windows [N,T,K]
test_windows [N,T,K]
test_labels_window [N]


In [13]:
# %% [STEP 3 - SMD] Cache windows per entity to disk (SELF-CONTAINED)

import os, gc, time, yaml
import numpy as np
from dataloader import load_dataset
from data_preprocess import generate_windows

# ----------------------------
# 1) Load configs again (so this cell works after kernel restart)
# ----------------------------
MODEL_CFG_FILE = "./mtad_gat.yaml"   # contains window_size, stride, batch_size, nb_epoch...
DATASET_CFG_FILE = "./SMD.yaml"      # contains Base + SMD dicts

assert os.path.exists(MODEL_CFG_FILE), f"Missing: {MODEL_CFG_FILE}"
assert os.path.exists(DATASET_CFG_FILE), f"Missing: {DATASET_CFG_FILE}"

with open(MODEL_CFG_FILE, "r") as f:
    model_cfg = yaml.safe_load(f)

with open(DATASET_CFG_FILE, "r") as f:
    dataset_cfg = yaml.safe_load(f)

# model params can be nested; handle both flat and nested
model_params = model_cfg.get("model", model_cfg) if isinstance(model_cfg, dict) else model_cfg
dataset_params = dataset_cfg

# ----------------------------
# 2) Read the right keys from your dataset YAML structure
# dataset_params = {"Base": {...}, "SMD": {"valid_ratio": 0}}
# ----------------------------
assert "Base" in dataset_params, f"dataset_params keys: {list(dataset_params.keys())}"
base = dataset_params["Base"]
smd  = dataset_params.get("SMD", {})

DATASET_NAME = str(base.get("dataset", "smd"))
valid_ratio = float(smd.get("valid_ratio", 0.0))

# IMPORTANT FIX: your EC2 data is in the repo
data_root = "./data/SMD/"
print("Resolved abs data_root:", os.path.abspath(data_root))
assert os.path.isdir(data_root), f"data_root not found: {data_root}"

# ----------------------------
# 3) Window params
# ----------------------------
window_size = int(model_params.get("window_size", 100))
stride = int(model_params.get("stride", 1))

# ----------------------------
# 4) Cache path
# ----------------------------
CACHE_DIR = os.path.join("./cache_windows", DATASET_NAME, f"T{window_size}_S{stride}")
os.makedirs(CACHE_DIR, exist_ok=True)

all_entities = base["entities"][:1]

print("\nDataset:", DATASET_NAME)
print("Caching to:", os.path.abspath(CACHE_DIR))
print("window_size:", window_size, "| stride:", stride, "| num_entities:", len(all_entities), "| valid_ratio:", valid_ratio)
print("postfix train/test/label:", base["train_postfix"], base["test_postfix"], base["test_label_postfix"])
print("dim:", int(base["dim"]))

# OPTIONAL: quick test only one entity
# all_entities = all_entities[:1]

# ----------------------------
# 5) Cache per entity
# ----------------------------
for i, entity in enumerate(all_entities):
    cache_path = os.path.join(CACHE_DIR, f"{entity}.npz")

    if os.path.exists(cache_path):
        print(f"[{i+1:02d}/{len(all_entities)}] {entity}: already cached -> skip")
        continue

    print(f"\n[{i+1:02d}/{len(all_entities)}] Caching entity={entity}")
    t0 = time.time()

    data_dict_1 = load_dataset(
        data_root=data_root,
        entities=[entity],
        valid_ratio=valid_ratio,
        dim=int(base["dim"]),
        test_label_postfix=base["test_label_postfix"],
        test_postfix=base["test_postfix"],
        train_postfix=base["train_postfix"],
        nrows=base.get("nrows", None),
    )

    # optional raw test label timeline, if provided by loader
    test_label_point = None
    try:
        test_label_point = data_dict_1[entity].get("test_label", None)
    except Exception:
        pass

    window_dict_1 = generate_windows(
        data_dict_1,
        window_size=window_size,
        stride=stride,
    )

    w = window_dict_1[entity]
    train_windows = w["train_windows"].astype(np.float32)  # [N_train, T, K]
    test_windows  = w["test_windows"].astype(np.float32)   # [N_test,  T, K]

    # windowed point-labels [N_test, T]
    if "test_label" in w:
        test_labels_point_window = w["test_label"]
    elif "test_labels" in w:
        test_labels_point_window = w["test_labels"]
    elif "label" in w:
        test_labels_point_window = w["label"]
    else:
        raise KeyError(f"Cannot find windowed labels in keys: {list(w.keys())}")

    test_labels_point_window = test_labels_point_window.astype(np.float32)
    test_labels_window = (test_labels_point_window.sum(axis=1) > 0).astype(np.int64)

    # starts (use tool-provided if present, else fallback)
    train_starts = w.get("train_starts", None)
    test_starts  = w.get("test_starts", None)
    if train_starts is None:
        train_starts = (np.arange(train_windows.shape[0], dtype=np.int64) * stride)
    else:
        train_starts = np.asarray(train_starts, dtype=np.int64)
    if test_starts is None:
        test_starts = (np.arange(test_windows.shape[0], dtype=np.int64) * stride)
    else:
        test_starts = np.asarray(test_starts, dtype=np.int64)

    np.savez_compressed(
        cache_path,
        train_windows=train_windows,
        test_windows=test_windows,
        test_labels_window=test_labels_window,
        test_labels_point_window=test_labels_point_window,
        train_starts=train_starts,
        test_starts=test_starts,
        test_label_point=test_label_point if test_label_point is not None else np.array([], dtype=np.int8),
        meta=np.array([DATASET_NAME, entity, window_size, stride], dtype=object),
    )

    anomaly_ratio = float(test_labels_window.mean()) if test_labels_window.size > 0 else 0.0
    print("  saved:", cache_path)
    print("  train:", train_windows.shape, "| test:", test_windows.shape,
          "| anomaly_ratio:", anomaly_ratio,
          "| elapsed_sec:", round(time.time() - t0, 2))

    del data_dict_1, window_dict_1, w, train_windows, test_windows, test_labels_point_window
    gc.collect()

Resolved abs data_root: /home/ec2-user/mtad-gat-independent/data/SMD

Dataset: smd
Caching to: /home/ec2-user/mtad-gat-independent/cache_windows/smd/T100_S1
window_size: 100 | stride: 1 | num_entities: 1 | valid_ratio: 0.0
postfix train/test/label: train.pkl test.pkl test_label.pkl
dim: 38
[01/1] machine-1-1: already cached -> skip


In [14]:
# %% [STEP 3 - SMD] Cache windows per entity to disk (SELF-CONTAINED)

import os, gc, time, yaml
import numpy as np
from dataloader import load_dataset
from data_preprocess import generate_windows

# ----------------------------
# 1) Load configs again (so this cell works after kernel restart)
# ----------------------------
MODEL_CFG_FILE = "./mtad_gat.yaml"   # contains window_size, stride, batch_size, nb_epoch...
DATASET_CFG_FILE = "./SMD.yaml"      # contains Base + SMD dicts

assert os.path.exists(MODEL_CFG_FILE), f"Missing: {MODEL_CFG_FILE}"
assert os.path.exists(DATASET_CFG_FILE), f"Missing: {DATASET_CFG_FILE}"

with open(MODEL_CFG_FILE, "r") as f:
    model_cfg = yaml.safe_load(f)

with open(DATASET_CFG_FILE, "r") as f:
    dataset_cfg = yaml.safe_load(f)

# model params can be nested; handle both flat and nested
model_params = model_cfg.get("model", model_cfg) if isinstance(model_cfg, dict) else model_cfg
dataset_params = dataset_cfg

# ----------------------------
# 2) Read the right keys from your dataset YAML structure
# dataset_params = {"Base": {...}, "SMD": {"valid_ratio": 0}}
# ----------------------------
assert "Base" in dataset_params, f"dataset_params keys: {list(dataset_params.keys())}"
base = dataset_params["Base"]
smd  = dataset_params.get("SMD", {})

DATASET_NAME = str(base.get("dataset", "smd"))
valid_ratio = float(smd.get("valid_ratio", 0.0))

# IMPORTANT FIX: your EC2 data is in the repo
data_root = "./data/SMD/"
print("Resolved abs data_root:", os.path.abspath(data_root))
assert os.path.isdir(data_root), f"data_root not found: {data_root}"

# ----------------------------
# 3) Window params
# ----------------------------
window_size = int(model_params.get("window_size", 100))
stride = int(model_params.get("stride", 1))

# ----------------------------
# 4) Cache path
# ----------------------------
CACHE_DIR = os.path.join("./cache_windows", DATASET_NAME, f"T{window_size}_S{stride}")
os.makedirs(CACHE_DIR, exist_ok=True)

all_entities = base["entities"][:1]

print("\nDataset:", DATASET_NAME)
print("Caching to:", os.path.abspath(CACHE_DIR))
print("window_size:", window_size, "| stride:", stride, "| num_entities:", len(all_entities), "| valid_ratio:", valid_ratio)
print("postfix train/test/label:", base["train_postfix"], base["test_postfix"], base["test_label_postfix"])
print("dim:", int(base["dim"]))

# OPTIONAL: quick test only one entity
# all_entities = all_entities[:1]

# ----------------------------
# 5) Cache per entity
# ----------------------------
for i, entity in enumerate(all_entities):
    cache_path = os.path.join(CACHE_DIR, f"{entity}.npz")

    if os.path.exists(cache_path):
        print(f"[{i+1:02d}/{len(all_entities)}] {entity}: already cached -> skip")
        continue

    print(f"\n[{i+1:02d}/{len(all_entities)}] Caching entity={entity}")
    t0 = time.time()

    data_dict_1 = load_dataset(
        data_root=data_root,
        entities=[entity],
        valid_ratio=valid_ratio,
        dim=int(base["dim"]),
        test_label_postfix=base["test_label_postfix"],
        test_postfix=base["test_postfix"],
        train_postfix=base["train_postfix"],
        nrows=base.get("nrows", None),
    )

    # optional raw test label timeline, if provided by loader
    test_label_point = None
    try:
        test_label_point = data_dict_1[entity].get("test_label", None)
    except Exception:
        pass

    window_dict_1 = generate_windows(
        data_dict_1,
        window_size=window_size,
        stride=stride,
    )

    w = window_dict_1[entity]
    train_windows = w["train_windows"].astype(np.float32)  # [N_train, T, K]
    test_windows  = w["test_windows"].astype(np.float32)   # [N_test,  T, K]

    # windowed point-labels [N_test, T]
    if "test_label" in w:
        test_labels_point_window = w["test_label"]
    elif "test_labels" in w:
        test_labels_point_window = w["test_labels"]
    elif "label" in w:
        test_labels_point_window = w["label"]
    else:
        raise KeyError(f"Cannot find windowed labels in keys: {list(w.keys())}")

    test_labels_point_window = test_labels_point_window.astype(np.float32)
    test_labels_window = (test_labels_point_window.sum(axis=1) > 0).astype(np.int64)

    # starts (use tool-provided if present, else fallback)
    train_starts = w.get("train_starts", None)
    test_starts  = w.get("test_starts", None)
    if train_starts is None:
        train_starts = (np.arange(train_windows.shape[0], dtype=np.int64) * stride)
    else:
        train_starts = np.asarray(train_starts, dtype=np.int64)
    if test_starts is None:
        test_starts = (np.arange(test_windows.shape[0], dtype=np.int64) * stride)
    else:
        test_starts = np.asarray(test_starts, dtype=np.int64)

    np.savez_compressed(
        cache_path,
        train_windows=train_windows,
        test_windows=test_windows,
        test_labels_window=test_labels_window,
        test_labels_point_window=test_labels_point_window,
        train_starts=train_starts,
        test_starts=test_starts,
        test_label_point=test_label_point if test_label_point is not None else np.array([], dtype=np.int8),
        meta=np.array([DATASET_NAME, entity, window_size, stride], dtype=object),
    )

    anomaly_ratio = float(test_labels_window.mean()) if test_labels_window.size > 0 else 0.0
    print("  saved:", cache_path)
    print("  train:", train_windows.shape, "| test:", test_windows.shape,
          "| anomaly_ratio:", anomaly_ratio,
          "| elapsed_sec:", round(time.time() - t0, 2))

    del data_dict_1, window_dict_1, w, train_windows, test_windows, test_labels_point_window
    gc.collect()

Resolved abs data_root: /home/ec2-user/mtad-gat-independent/data/SMD

Dataset: smd
Caching to: /home/ec2-user/mtad-gat-independent/cache_windows/smd/T100_S1
window_size: 100 | stride: 1 | num_entities: 1 | valid_ratio: 0.0
postfix train/test/label: train.pkl test.pkl test_label.pkl
dim: 38
[01/1] machine-1-1: already cached -> skip


Raw time-series
      ↓
Step 3 → Window cache (disk)
      ↓
Step 4 → DataLoader (reshape only)
      ↓
Step 5 → Build MTGFlow model
      ↓
Step 6 → Train model (maximize likelihood)
      ↓
Evaluation → Compute anomaly scores + AUROC

## A) Learning mode — train ONE entity interactively

In [15]:
# %% [STEP 4] Setup paths + imports (MTGFlow)

import os, sys
import numpy as np
import pandas as pd
import torch

# ---- Set this to your MTGFlow repo path (folder that contains models/MTGFLOW.py)
# Example: MTGFLOW_REPO_PATH = "./MTGFLOW-main"
MTGFLOW_REPO_PATH = "./MTGFLOW-main"
assert os.path.isdir(MTGFLOW_REPO_PATH), f"MTGFlow repo not found: {MTGFLOW_REPO_PATH}"

if MTGFLOW_REPO_PATH not in sys.path:
    sys.path.insert(0, MTGFLOW_REPO_PATH)

from models.MTGFLOW import MTGFLOW

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ DEVICE:", DEVICE)
print("✅ MTGFLOW_REPO_PATH:", os.path.abspath(MTGFLOW_REPO_PATH))

✅ DEVICE: cuda
✅ MTGFLOW_REPO_PATH: /home/ec2-user/mtad-gat-independent/MTGFLOW-main


## STEP 5 — Helper functions: dataset, training, evaluation, saving metrics

In [16]:
# %% [STEP 5] Shared helpers: loaders + train + evaluate + save metrics (MTGFlow)

import os, json, gc, time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_value_

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    f1_score,
    precision_score,
    recall_score,
)

# ---------- Results root ----------
RESULTS_ROOT = os.path.join("./results_mtgflow", DATASET_NAME, f"T{window_size}_S{stride}")
os.makedirs(RESULTS_ROOT, exist_ok=True)
print("✅ RESULTS_ROOT:", os.path.abspath(RESULTS_ROOT))

# ---------- Training Hyperparams ----------
# You can override these safely:
LR = float(model_params.get("lr", 1e-3)) if isinstance(model_params, dict) else 1e-3
BATCH_SIZE = int(model_params.get("batch_size", 512))
EPOCHS = int(model_params.get("nb_epoch", 10))
GRAD_CLIP = 1.0
WEIGHT_DECAY = 0.0

print("Hyperparams:", dict(lr=LR, batch_size=BATCH_SIZE, epochs=EPOCHS))


# ---------- Cache loader ----------
def load_cached_entity(entity: str):
    cache_path = os.path.join(CACHE_DIR, f"{entity}.npz")
    assert os.path.exists(cache_path), f"Missing cache: {cache_path}"
    d = np.load(cache_path, allow_pickle=True)
    return {
        "cache_path": cache_path,
        "train_windows": d["train_windows"],  # [N,T,K]
        "test_windows": d["test_windows"],    # [N,T,K]
        "test_labels_window": d["test_labels_window"],  # [N]
        "test_labels_point_window": d["test_labels_point_window"],  # [N,T]
        "train_starts": d["train_starts"],
        "test_starts": d["test_starts"],
        "test_label_point": d["test_label_point"] if "test_label_point" in d.files else np.array([], dtype=np.int8),
        "meta": d["meta"] if "meta" in d.files else None,
    }


# ---------- Dataset ----------
class MTGFlowWindowDataset(Dataset):
    """
    Cached windows: [N, T, K]
    MTGFlow expects: [K, T, 1]
    """
    def __init__(self, windows, labels=None, starts=None):
        self.windows = windows.astype(np.float32)
        self.labels = None if labels is None else labels.astype(np.int64)
        self.starts = None if starts is None else starts.astype(np.int64)

    def __len__(self):
        return self.windows.shape[0]

    def __getitem__(self, idx):
        w = self.windows[idx]                      # [T,K]
        x = torch.from_numpy(w).transpose(0, 1)    # [K,T]
        x = x.unsqueeze(-1)                        # [K,T,1]
        y = 0 if self.labels is None else int(self.labels[idx])
        s = -1 if self.starts is None else int(self.starts[idx])
        return x, y, int(idx), s


def make_loaders(pack):
    train_ds = MTGFlowWindowDataset(pack["train_windows"], labels=None, starts=pack["train_starts"])
    test_ds  = MTGFlowWindowDataset(pack["test_windows"],  labels=pack["test_labels_window"], starts=pack["test_starts"])

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
    return train_loader, test_loader


# ---------- Metrics helpers ----------
def best_f1_threshold(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    if thresholds.size == 0:
        return {"best_threshold": None, "best_f1": np.nan, "best_precision": np.nan, "best_recall": np.nan}

    p = precision[:-1]
    r = recall[:-1]
    f1 = (2 * p * r) / (p + r + 1e-12)
    best_idx = int(np.nanargmax(f1))
    return {
        "best_threshold": float(thresholds[best_idx]),
        "best_f1": float(f1[best_idx]),
        "best_precision": float(p[best_idx]),
        "best_recall": float(r[best_idx]),
    }


def windows_to_point_scores(window_scores, window_starts, series_len, window_size, agg="max"):
    if agg == "max":
        ps = np.full(series_len, -np.inf, dtype=np.float64)
    else:
        ps = np.zeros(series_len, dtype=np.float64)
        cnt = np.zeros(series_len, dtype=np.int64)

    for s, sc in zip(window_starts, window_scores):
        s = int(s)
        e = min(series_len, s + int(window_size))
        if e <= s:
            continue
        if agg == "max":
            ps[s:e] = np.maximum(ps[s:e], float(sc))
        else:
            ps[s:e] += float(sc)
            cnt[s:e] += 1

    if agg == "max":
        uncovered = np.isneginf(ps)
        if np.any(uncovered):
            ps[uncovered] = float(np.min(window_scores)) if window_scores.size else 0.0
        return ps

    covered = cnt > 0
    ps[covered] /= cnt[covered]
    ps[~covered] = float(np.min(window_scores)) if window_scores.size else 0.0
    return ps


def windows_to_point_labels(test_labels_point_window, test_starts, series_len, window_size):
    pl = np.zeros(series_len, dtype=np.int8)
    for i, s in enumerate(test_starts):
        s = int(s)
        e = min(series_len, s + int(window_size))
        if e <= s:
            continue
        wlab = test_labels_point_window[i][: (e - s)]
        pl[s:e] = np.maximum(pl[s:e], (wlab > 0).astype(np.int8))
    return pl


def topk_windows(window_scores, window_starts, window_size, k=20):
    idx = np.argsort(-window_scores)[:k]
    out = []
    for rank, i in enumerate(idx, start=1):
        s = int(window_starts[i])
        out.append({"rank": rank, "window_index": int(i), "start": s, "end": int(s + window_size), "score": float(window_scores[i])})
    return out


# ---------- Model builder ----------
def build_mtgflow_model(K, T, n_blocks=2, hidden_size=32, n_hidden=1, dropout=0.0, model_type="MAF", batch_norm=False):
    model = MTGFLOW(
        n_blocks=n_blocks,
        input_size=1,
        hidden_size=hidden_size,
        n_hidden=n_hidden,
        window_size=T,
        n_sensor=K,
        dropout=dropout,
        model=model_type,
        batch_norm=batch_norm,
    ).to(DEVICE)
    return model


# ---------- Evaluation ----------
def eval_window_scores(model, test_loader):
    model.eval()
    scores, labels, starts = [], [], []
    with torch.no_grad():
        for x, y, _, s in test_loader:
            x = x.to(DEVICE)
            nll = (-model.test(x)).detach().cpu().numpy().reshape(-1)  # NLL score
            scores.append(nll)
            labels.append(np.asarray(y, dtype=np.int64).reshape(-1))
            starts.append(np.asarray(s, dtype=np.int64).reshape(-1))
    return (np.concatenate(scores) if scores else np.array([]),
            np.concatenate(labels) if labels else np.array([]),
            np.concatenate(starts) if starts else np.array([]))


def compute_and_save_results(entity, pack, window_scores, window_labels, window_starts, out_dir, topk=20):
    os.makedirs(out_dir, exist_ok=True)

    y = window_labels.astype(int)
    s = window_scores.astype(float)

    # window metrics
    auroc_w = roc_auc_score(y, s) if y.size and np.unique(y).size > 1 else float("nan")
    aupr_w  = average_precision_score(y, s) if y.size and np.unique(y).size > 1 else float("nan")

    bf1 = best_f1_threshold(y, s)
    thr = bf1["best_threshold"]

    if thr is None:
        yhat = np.zeros_like(y)
        f1_w = p_w = r_w = float("nan")
    else:
        yhat = (s >= thr).astype(int)
        f1_w = float(f1_score(y, yhat, zero_division=0))
        p_w  = float(precision_score(y, yhat, zero_division=0))
        r_w  = float(recall_score(y, yhat, zero_division=0))

    # time localization
    # Prefer raw point timeline label if available; else reconstruct from windowed labels
    test_label_point = pack.get("test_label_point", np.array([], dtype=np.int8))
    if test_label_point.size > 0:
        series_len = int(test_label_point.shape[0])
        point_labels = (test_label_point > 0).astype(np.int8)
    else:
        # reconstruct length from windows
        series_len = int(window_starts.max() + window_size) if window_starts.size else int(window_size)
        point_labels = windows_to_point_labels(pack["test_labels_point_window"], window_starts, series_len, window_size)

    point_scores_max  = windows_to_point_scores(s, window_starts, series_len, window_size, agg="max")
    point_scores_mean = windows_to_point_scores(s, window_starts, series_len, window_size, agg="mean")

    auroc_p = roc_auc_score(point_labels, point_scores_max) if point_labels.size and np.unique(point_labels).size > 1 else float("nan")
    aupr_p  = average_precision_score(point_labels, point_scores_max) if point_labels.size and np.unique(point_labels).size > 1 else float("nan")

    topk_list = topk_windows(s, window_starts, window_size, k=topk)

    metrics = {
        "dataset": DATASET_NAME,
        "entity": entity,
        "window_size": int(window_size),
        "stride": int(stride),
        "num_windows": int(s.size),
        "positive_windows": int(y.sum()) if y.size else 0,
        "anomaly_ratio_window": float(y.mean()) if y.size else 0.0,

        "auroc_window": float(auroc_w),
        "aupr_window": float(aupr_w),

        "best_threshold_f1_window": thr,
        "best_f1_window": float(bf1["best_f1"]),
        "best_precision_window": float(bf1["best_precision"]),
        "best_recall_window": float(bf1["best_recall"]),

        "f1_at_best_thr_window": float(f1_w),
        "precision_at_best_thr_window": float(p_w),
        "recall_at_best_thr_window": float(r_w),

        "series_len": int(series_len),
        "auroc_point_max": float(auroc_p),
        "aupr_point_max": float(aupr_p),

        "topk_windows": topk_list,
    }

    # Save metrics
    metrics_json = os.path.join(out_dir, f"{entity}_metrics.json")
    with open(metrics_json, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    metrics_csv = os.path.join(out_dir, f"{entity}_metrics.csv")
    pd.DataFrame([metrics]).to_csv(metrics_csv, index=False)

    # Save arrays
    np.savez_compressed(
        os.path.join(out_dir, f"{entity}_scores_windows.npz"),
        window_scores=s.astype(np.float32),
        window_labels=y.astype(np.int8),
        window_pred=yhat.astype(np.int8),
        window_starts=window_starts.astype(np.int32),
    )

    np.savez_compressed(
        os.path.join(out_dir, f"{entity}_scores_points.npz"),
        point_scores_max=point_scores_max.astype(np.float32),
        point_scores_mean=point_scores_mean.astype(np.float32),
        point_labels=point_labels.astype(np.int8),
    )

    # Print summary
    print(
        f"[{entity}] AUROC(w)={auroc_w:.4f} AUPR(w)={aupr_w:.4f} "
        f"BestF1={metrics['best_f1_window']:.4f} thr={thr} | "
        f"AUROC(p,max)={auroc_p:.4f} AUPR(p,max)={aupr_p:.4f}"
    )
    print("  saved ->", out_dir)

    return metrics


# ---------- Train one entity ----------
def train_one_entity(entity, n_blocks=2, hidden_size=32, n_hidden=1, dropout=0.0, model_type="MAF", batch_norm=False):
    pack = load_cached_entity(entity)
    out_dir = os.path.join(RESULTS_ROOT, entity)
    os.makedirs(os.path.join(out_dir, "models"), exist_ok=True)
    os.makedirs(os.path.join(out_dir, "history"), exist_ok=True)

    train_loader, test_loader = make_loaders(pack)

    K = int(pack["train_windows"].shape[-1])
    T = int(pack["train_windows"].shape[1])

    model = build_mtgflow_model(
        K, T,
        n_blocks=n_blocks,
        hidden_size=hidden_size,
        n_hidden=n_hidden,
        dropout=dropout,
        model_type=model_type,
        batch_norm=batch_norm
    )

    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    history = {"epoch": [], "train_nll": [], "auroc_window": [], "aupr_window": []}
    best_auroc = -1.0
    best_state = None

    for ep in range(1, EPOCHS + 1):
        model.train()
        losses = []

        for x, _, _, _ in train_loader:
            x = x.to(DEVICE)
            opt.zero_grad()
            loss = -model(x)     # maximize log-likelihood => minimize NLL
            loss.backward()
            clip_grad_value_(model.parameters(), GRAD_CLIP)
            opt.step()
            losses.append(float(loss.item()))

        train_nll = float(np.mean(losses)) if losses else float("nan")

        # quick eval per epoch (window-level only)
        scores, labels_w, starts = eval_window_scores(model, test_loader)
        if labels_w.size and np.unique(labels_w).size > 1:
            auroc_w = float(roc_auc_score(labels_w, scores))
            aupr_w  = float(average_precision_score(labels_w, scores))
        else:
            auroc_w, aupr_w = float("nan"), float("nan")

        history["epoch"].append(ep)
        history["train_nll"].append(train_nll)
        history["auroc_window"].append(auroc_w)
        history["aupr_window"].append(aupr_w)

        print(f"  epoch {ep:03d}/{EPOCHS} | train_nll={train_nll:.4f} | AUROC(w)={auroc_w:.4f} | AUPR(w)={aupr_w:.4f}")

        if np.isfinite(auroc_w) and auroc_w > best_auroc:
            best_auroc = auroc_w
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # restore best
    if best_state is not None:
        model.load_state_dict(best_state)

    # final eval + save
    scores, labels_w, starts = eval_window_scores(model, test_loader)

    # save model
    ckpt_path = os.path.join(out_dir, "models", f"{entity}.pt")
    torch.save(
        {
            "state_dict": model.state_dict(),
            "dataset": DATASET_NAME,
            "entity": entity,
            "K": K, "T": T,
            "window_size": int(window_size),
            "stride": int(stride),
            "n_blocks": int(n_blocks),
            "hidden_size": int(hidden_size),
            "n_hidden": int(n_hidden),
            "dropout": float(dropout),
            "model_type": str(model_type),
            "batch_norm": bool(batch_norm),
        },
        ckpt_path
    )

    # save history
    hist_path = os.path.join(out_dir, "history", f"{entity}_history.csv")
    pd.DataFrame(history).to_csv(hist_path, index=False)

    # metrics + arrays
    metrics = compute_and_save_results(
        entity=entity,
        pack=pack,
        window_scores=scores,
        window_labels=labels_w,
        window_starts=starts,
        out_dir=out_dir,
        topk=20
    )
    metrics["best_auroc_during_training"] = float(best_auroc)
    metrics["ckpt_path"] = ckpt_path
    metrics["history_path"] = hist_path

    # cleanup
    del model, opt
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return metrics

✅ RESULTS_ROOT: /home/ec2-user/mtad-gat-independent/results_mtgflow/smd/T100_S1
Hyperparams: {'lr': 0.001, 'batch_size': 512, 'epochs': 10}


### STEP 6 — Run for ONE entity or ALL entities
Paste this as the next cell. You can switch RUN_ALL = False/True.

In [18]:
# %% [STEP 6] Train/Eval one entity or all entities + save global summary

RUN_ALL = False        # True => train all entities
ENTITY_TO_RUN = all_entities[0]   # pick one (your step 3 uses [:1] anyway)

# If you want all entities, set:
# all_entities = base["entities"]   (in your Step 3 cell)

summaries = []

if RUN_ALL:
    entities = list(base["entities"])
else:
    entities = [ENTITY_TO_RUN]

print("Entities to run:", entities)

for i, ent in enumerate(entities, start=1):
    print(f"\n=== [{i}/{len(entities)}] Training entity: {ent} ===")
    m = train_one_entity(
        ent,
        n_blocks=2,
        hidden_size=32,
        n_hidden=1,
        dropout=0.0,
        model_type="MAF",
        batch_norm=False,
    )
    summaries.append(m)

summary_df = pd.DataFrame(summaries)
overall_csv = os.path.join(RESULTS_ROOT, "_ALL_ENTITIES_SUMMARY.csv")
summary_df.to_csv(overall_csv, index=False)

print("\n✅ Saved overall summary:", overall_csv)
summary_df

Entities to run: ['machine-1-1']

=== [1/1] Training entity: machine-1-1 ===
  epoch 001/10 | train_nll=-1.1559 | AUROC(w)=0.5748 | AUPR(w)=0.3496
  epoch 002/10 | train_nll=-4.4913 | AUROC(w)=0.6730 | AUPR(w)=0.3618
  epoch 003/10 | train_nll=-5.4848 | AUROC(w)=0.5991 | AUPR(w)=0.3500
  epoch 004/10 | train_nll=-2.1191 | AUROC(w)=0.8649 | AUPR(w)=0.4517
  epoch 005/10 | train_nll=-6.3247 | AUROC(w)=0.5515 | AUPR(w)=0.3413
  epoch 006/10 | train_nll=-3.8166 | AUROC(w)=0.5080 | AUPR(w)=0.2550
  epoch 007/10 | train_nll=-5.0899 | AUROC(w)=0.7015 | AUPR(w)=0.2285
  epoch 008/10 | train_nll=-6.1050 | AUROC(w)=0.6670 | AUPR(w)=0.1847
  epoch 009/10 | train_nll=-5.7747 | AUROC(w)=0.8787 | AUPR(w)=0.5312
  epoch 010/10 | train_nll=-4.9462 | AUROC(w)=0.1660 | AUPR(w)=0.0775
[machine-1-1] AUROC(w)=0.8787 AUPR(w)=0.5312 BestF1=0.5803 thr=-6.906196594238281 | AUROC(p,max)=0.9239 AUPR(p,max)=0.5506
  saved -> ./results_mtgflow/smd/T100_S1/machine-1-1

✅ Saved overall summary: ./results_mtgflow/smd

,dataset,entity,window_size,stride,num_windows,positive_windows,anomaly_ratio_window,auroc_window,aupr_window,best_threshold_f1_window,...,f1_at_best_thr_window,precision_at_best_thr_window,recall_at_best_thr_window,series_len,auroc_point_max,aupr_point_max,topk_windows,best_auroc_during_training,ckpt_path,history_path
0,smd,machine-1-1,100,1,28379,3486,0.122837,0.87868,0.531214,-6.906197,...,0.580345,0.554974,0.608147,28479,0.923942,0.550608,"[{'rank': 1, 'window_index': 17417, 'start': 1...",0.87868,./results_mtgflow/smd/T100_S1/machine-1-1/mode...,./results_mtgflow/smd/T100_S1/machine-1-1/hist...
